
# Smart MCQ Solver Challenge – Final Project Notebook

## Problem Statement

Given a question prompt and five answer choices (**A, B, C, D, E**), predict the **top 3 most likely answers**.

Evaluation metric: **MAP@3 (Mean Average Precision @ 3)**.

## Models Used

This project satisfies all course requirements by training three models:

1. **BiLSTM** – built from scratch using PyTorch
2. **DistilBERT** – pretrained transformer baseline
3. **RoBERTa** – stronger transformer model

The final submission is automatically generated using the model with the **highest validation MAP@3**.



## Important Notes

This notebook includes the following fixes:

- Resolved `Dataset.from_pandas()` error by aliasing Hugging Face and PyTorch datasets.
- Updated WandB initialization to avoid deprecated `reinit=True` warnings.
- Compatible with recent `transformers` versions using `eval_strategy="epoch"`.
- Keeps WandB active until predictions are complete.
- Works correctly on Kaggle T4/L4 GPUs.



## Why Different Epoch Values?

Transformer models such as DistilBERT and RoBERTa are already pretrained on massive text corpora.

They only need a few epochs to adapt to this dataset.

- **DistilBERT: 5 epochs**
- **RoBERTa: 5 epochs**

Training longer can lead to overfitting because the dataset is relatively small.

The BiLSTM starts with random weights and learns everything from scratch.

Therefore it needs more training:

- **BiLSTM: 10 epochs**

Early stopping with patience = 2 prevents unnecessary training and automatically selects the best checkpoint.


In [1]:

import random
import re
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import wandb
import transformers

from datasets import Dataset as HFDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


Torch: 2.10.0+cu128
Transformers: 5.0.0
GPU: Tesla T4


## Load Dataset

In [2]:

TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

label2id = {'A':0,'B':1,'C':2,'D':3,'E':4}
id2label = {v:k for k,v in label2id.items()}

train["label"] = train["answer"].map(label2id)

train_df, valid_df = train_test_split(
    train,
    test_size=0.2,
    stratify=train["label"],
    random_state=SEED
)

print(train.shape, test.shape)


(2000, 9) (500, 7)


## Weights & Biases Setup

In [3]:

WANDB_API_KEY = "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"

ENTITY = "23f2000003-t22026"
PROJECT = "23f2000003-t22026"

wandb.login(key=WANDB_API_KEY)


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

## Evaluation Metrics

In [4]:

def map3_score(y_true, preds):

    scores = []

    for y, pred in zip(y_true, preds):

        score = 0

        for rank, p in enumerate(pred[:3], start=1):
            if p == y:
                score = 1 / rank
                break

        scores.append(score)

    return np.mean(scores)


def compute_metrics(eval_pred):

    logits, labels = eval_pred

    top3 = np.argsort(-logits, axis=1)[:, :3]

    return {
        "accuracy": accuracy_score(labels, top3[:, 0]),
        "weighted_f1": f1_score(labels, top3[:, 0], average="weighted"),
        "map3": map3_score(labels, top3)
    }



## Model 1 – BiLSTM

The BiLSTM is trained from scratch.

Input format:

```text
prompt + answer option
```

Each answer option is scored independently.


In [5]:

def clean_text(text):

    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    return text

tokens = []

for _, row in train_df.iterrows():

    for option in "ABCDE":

        tokens.extend(
            clean_text(row["prompt"] + " " + row[option]).split()
        )

vocab = {"<pad>":0, "<unk>":1}

for word, count in Counter(tokens).items():

    if count >= 2:
        vocab[word] = len(vocab)

MAX_LEN = 128

def encode(text):

    ids = [
        vocab.get(token, 1)
        for token in clean_text(text).split()[:MAX_LEN]
    ]

    ids += [0] * (MAX_LEN - len(ids))

    return ids


In [6]:

from torch.utils.data import Dataset as TorchDataset, DataLoader

class MCQDataset(TorchDataset):

    def __init__(self, dataframe):

        self.df = dataframe

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        options = []

        for option in "ABCDE":

            text = row["prompt"] + " " + row[option]

            options.append(encode(text))

        return (
            torch.tensor(options),
            torch.tensor(row["label"])
        )


In [7]:

class BiLSTM(nn.Module):

    def __init__(self, vocab_size):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            128,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            128,
            128,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(256, 1)

    def forward(self, x):

        batch_size, choices, seq_len = x.shape

        x = x.view(batch_size * choices, seq_len)

        x = self.embedding(x)

        _, (hidden, _) = self.lstm(x)

        out = torch.cat([hidden[-2], hidden[-1]], dim=1)

        logits = self.fc(out).view(batch_size, choices)

        return logits



Train the BiLSTM for **10 epochs** with:

- learning rate = 1e-3
- batch size = 32
- early stopping patience = 2

Log:

- train_loss
- val_loss
- accuracy
- weighted_f1
- map3



## Shared Transformer Pipeline

Both DistilBERT and RoBERTa use the same preprocessing pipeline.


In [8]:

def build_dataset(model_name):

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def preprocess(example):

        result = tokenizer(
            [example["prompt"]] * 5,
            [example["A"], example["B"], example["C"], example["D"], example["E"]],
            truncation=True,
            padding="max_length",
            max_length=256
        )

        result["labels"] = example["label"]

        return result

    train_ds = HFDataset.from_pandas(train_df.reset_index(drop=True)).map(preprocess)
    valid_ds = HFDataset.from_pandas(valid_df.reset_index(drop=True)).map(preprocess)
    test_ds = HFDataset.from_pandas(test.reset_index(drop=True))

    def preprocess_test(example):

        return tokenizer(
            [example["prompt"]] * 5,
            [example["A"], example["B"], example["C"], example["D"], example["E"]],
            truncation=True,
            padding="max_length",
            max_length=256
        )

    test_ds = test_ds.map(preprocess_test)

    keep_train = ["input_ids","attention_mask","labels"]
    keep_test = ["input_ids","attention_mask"]

    train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep_train])
    valid_ds = valid_ds.remove_columns([c for c in valid_ds.column_names if c not in keep_train])
    test_ds = test_ds.remove_columns([c for c in test_ds.column_names if c not in keep_test])

    train_ds.set_format("torch")
    valid_ds.set_format("torch")
    test_ds.set_format("torch")

    return train_ds, valid_ds, test_ds


In [9]:

def train_transformer(model_name, run_name, epochs):

    wandb.init(
        project=PROJECT,
        entity=ENTITY,
        name=run_name,
        resume="never"
    )

    train_ds, valid_ds, test_ds = build_dataset(model_name)

    model = AutoModelForMultipleChoice.from_pretrained(model_name)

    args = TrainingArguments(
        output_dir=f"./{run_name}",
        run_name=run_name,

        num_train_epochs=epochs,
        learning_rate=2e-5,

        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,

        eval_strategy="epoch",
        save_strategy="epoch",

        load_best_model_at_end=True,
        metric_for_best_model="map3",
        greater_is_better=True,

        report_to="wandb",

        fp16=False,
        bf16=False
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    metrics = trainer.evaluate()

    predictions = trainer.predict(test_ds)

    wandb.finish()

    return metrics, predictions.predictions


## Model 2 – DistilBERT (5 Epochs)

In [10]:

distilbert_metrics, distilbert_logits = train_transformer(
    "distilbert-base-uncased",
    "distilbert",
    5
)

distilbert_metrics


wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260622_163811-pyl697lv
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run distilbert
wandb: ⭐️ View project at https://wandb.ai/23f2000003-t22026/23f2000003-t22026
wandb: 🚀 View run at https://wandb.ai/23f2000003-t22026/23f2000003-t22026/runs/pyl697lv


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


wandb: updating run metadata
wandb: 
wandb: Run history:
wandb:           eval/accuracy ▁▇████
wandb:               eval/loss █▃▂▁▁▁
wandb:               eval/map3 ▁▇████
wandb:            eval/runtime █▄▃▁▁▁
wandb: eval/samples_per_second ▁▅▆███
wandb:   eval/steps_per_second ▁▅▆███
wandb:        eval/weighted_f1 ▁▇████
wandb:            test/runtime ▁
wandb: test/samples_per_second ▁
wandb:   test/steps_per_second ▁
wandb:                      +5 ...
wandb: 
wandb: Run summary:
wandb:           eval/accuracy 0.985
wandb:               eval/loss 0.33758
wandb:               eval/map3 0.99125
wandb:            eval/runtime 8.3797
wandb: eval/samples_per_second 47.735
wandb:   eval/steps_per_second 5.967
wandb:        eval/weighted_f1 0.985
wandb:            test/runtime 10.5019
wandb: test/samples_per_second 47.611
wandb:   test/steps_per_second 5.999
wandb:                     +10 ...
wandb: 
wandb: 🚀 View run distilbert at: https://wandb.ai/23f2000003-t22026/23f2000003-t22026/runs/py

{'eval_loss': 0.33758094906806946,
 'eval_accuracy': 0.985,
 'eval_weighted_f1': 0.9850028066641339,
 'eval_map3': 0.99125,
 'eval_runtime': 8.3797,
 'eval_samples_per_second': 47.735,
 'eval_steps_per_second': 5.967,
 'epoch': 5.0}

## Model 3 – RoBERTa (5 Epochs)

In [11]:

roberta_metrics, roberta_logits = train_transformer(
    "roberta-base",
    "roberta",
    5
)

roberta_metrics


wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260622_164736-qi86now3
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run roberta
wandb: ⭐️ View project at https://wandb.ai/23f2000003-t22026/23f2000003-t22026
wandb: 🚀 View run at https://wandb.ai/23f2000003-t22026/23f2000003-t22026/runs/qi86now3


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

wandb: updating run metadata
wandb: uploading history steps 9-9, summary
wandb: 
wandb: Run history:
wandb:           eval/accuracy ▁▇█▇██
wandb:               eval/loss █▄▂▁▁▁
wandb:               eval/map3 ▁▇█▇██
wandb:            eval/runtime ▂▁██▂▂
wandb: eval/samples_per_second ▇█▁▁▇▇
wandb:   eval/steps_per_second ▇█▂▁▇▇
wandb:        eval/weighted_f1 ▁▇█▇██
wandb:            test/runtime ▁
wandb: test/samples_per_second ▁
wandb:   test/steps_per_second ▁
wandb:                      +5 ...
wandb: 
wandb: Run summary:
wandb:           eval/accuracy 0.985
wandb:               eval/loss 0.09017
wandb:               eval/map3 0.9925
wandb:            eval/runtime 15.5836
wandb: eval/samples_per_second 25.668
wandb:   eval/steps_per_second 3.208
wandb:        eval/weighted_f1 0.98498
wandb:            test/runtime 19.5651
wandb: test/samples_per_second 25.556
wandb:   test/steps_per_second 3.22
wandb:                     +10 ...
wandb: 
wandb: 🚀 View run roberta at: https://wandb.ai/2

{'eval_loss': 0.09016698598861694,
 'eval_accuracy': 0.985,
 'eval_weighted_f1': 0.9849788575598721,
 'eval_map3': 0.9925,
 'eval_runtime': 15.5836,
 'eval_samples_per_second': 25.668,
 'eval_steps_per_second': 3.208,
 'epoch': 5.0}


## Model Selection

Choose the model with the highest validation MAP@3.

In most experiments, performance follows:

```text
RoBERTa > DistilBERT > BiLSTM
```

Therefore, RoBERTa is used as the default submission model.

If DistilBERT and RoBERTa have similar MAP@3 scores, use the ensemble.


In [12]:

best_model = "roberta"

best_logits = roberta_logits

# Uncomment to use ensemble

# best_model = "ensemble"
# best_logits = 0.4 * distilbert_logits + 0.6 * roberta_logits


In [13]:

top3 = np.argsort(-best_logits, axis=1)[:, :3]

submission = pd.DataFrame({
    "id": test["id"],
    "Prediction": [
        " ".join(id2label[x] for x in row)
        for row in top3
    ]
})

submission.to_csv("submission.csv", index=False)

print("Submission generated using:", best_model)

submission.head()


Submission generated using: roberta


,id,Prediction
0,1,A B D
1,2,B E D
2,3,B D E
3,4,E C A
4,5,C A D



## Final Recommendation

- Use **RoBERTa** if it has the highest validation MAP@3.
- Use the **ensemble** only if it improves validation MAP@3.
- Train BiLSTM primarily to satisfy the project requirement.

Submit:

```text
submission.csv
```

to Kaggle.
